In [16]:
import numpy as np
from scipy.sparse.linalg import eigs, LinearOperator
from scipy.linalg import expm
from typing import Tuple
import warnings
warnings.filterwarnings("ignore")

def define_spin1_afh_hamiltonian() -> np.ndarray:
    """
    Constructs the nearest-neighbor Spin-1 AFH Hamiltonian: H = S^x S^x + S^y S^y + S^z S^z
    Returns a rank-4 tensor (3, 3, 3, 3).
    """
    Sz = np.array([[1, 0, 0], 
                   [0, 0, 0], 
                   [0, 0, -1]], dtype=complex)
    Sp = np.array([[0, np.sqrt(2), 0], 
                   [0, 0, np.sqrt(2)], 
                   [0, 0, 0]], dtype=complex)
    Sm = Sp.T
    
    Sx = (Sp + Sm) / 2.0
    Sy = (Sp - Sm) / 2.0j
    
    H_local_matrix = (np.kron(Sx, Sx) + np.kron(Sy, Sy) + np.kron(Sz, Sz)).real
    return H_local_matrix.reshape(3, 3, 3, 3)

def initialize_uniform_mps(bond_dim: int = 64, phys_dim: int = 3) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Initializes a 2-site unit cell with slight noise to prevent divergence."""
    GammaA = np.zeros((bond_dim, phys_dim, bond_dim), dtype=complex)
    GammaB = np.zeros((bond_dim, phys_dim, bond_dim), dtype=complex)
    
    GammaA[0, 1, 0] = 1.0  
    GammaB[0, 1, 0] = 1.0  
    
    noise = 1e-2
    GammaA += noise * (np.random.randn(bond_dim, phys_dim, bond_dim) + 1j * np.random.randn(bond_dim, phys_dim, bond_dim))
    GammaB += noise * (np.random.randn(bond_dim, phys_dim, bond_dim) + 1j * np.random.randn(bond_dim, phys_dim, bond_dim))
    
    GammaA /= np.linalg.norm(GammaA)
    GammaB /= np.linalg.norm(GammaB)
    
    LambdaA = np.ones(bond_dim) / np.sqrt(bond_dim)
    LambdaB = np.ones(bond_dim) / np.sqrt(bond_dim)
    
    return GammaA, GammaB, LambdaA, LambdaB

def calculate_local_energy(GammaA, GammaB, LambdaA, LambdaB, h_local) -> float:
    """Calculates energy density across the 2-site unit cell."""
    # A-B bond
    ThetaAB = np.einsum('a, asb, b, btc, c -> astc', LambdaB, GammaA, LambdaA, GammaB, LambdaB, optimize=True)
    E_AB = np.einsum('astc, STst, aSTc -> ', np.conj(ThetaAB), h_local, ThetaAB, optimize=True).real
    norm_AB = np.einsum('astc, astc -> ', np.conj(ThetaAB), ThetaAB, optimize=True).real
    
    # B-A bond
    ThetaBA = np.einsum('a, asb, b, btc, c -> astc', LambdaA, GammaB, LambdaB, GammaA, LambdaA, optimize=True)
    E_BA = np.einsum('astc, STst, aSTc -> ', np.conj(ThetaBA), h_local, ThetaBA, optimize=True).real
    norm_BA = np.einsum('astc, astc -> ', np.conj(ThetaBA), ThetaBA, optimize=True).real
    
    return 0.5 * (E_AB / norm_AB + E_BA / norm_BA)

def run_itebd_2site(GammaA, GammaB, LambdaA, LambdaB, gate) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Performs a single rigorous 2-site iTEBD update step."""
    D, d, _ = GammaA.shape
    cutoff = 1e-12 
    
    # --- Update A-B Bond ---
    ThetaAB = np.einsum('a, asb, b, btc, c -> astc', LambdaB, GammaA, LambdaA, GammaB, LambdaB, optimize=True)
    ThetaAB_evolved = np.einsum('STst, astc -> aSTc', gate, ThetaAB, optimize=True)
    
    U, S, Vh = np.linalg.svd(ThetaAB_evolved.reshape(D * d, d * D), full_matrices=False)
    U_trunc = U[:, :D].reshape(D, d, D)
    S_trunc = S[:D] / np.linalg.norm(S[:D])
    Vh_trunc = Vh[:D, :].reshape(D, d, D)
    
    inv_LambdaB = np.zeros_like(LambdaB)
    valid = LambdaB > cutoff
    inv_LambdaB[valid] = 1.0 / LambdaB[valid]
    L_invB = np.diag(inv_LambdaB)
    
    GammaA_new = np.einsum('ab, bsc -> asc', L_invB, U_trunc, optimize=True)
    GammaB_new = np.einsum('asb, bc -> asc', Vh_trunc, L_invB, optimize=True)
    LambdaA_new = S_trunc

    # --- Update B-A Bond ---
    ThetaBA = np.einsum('a, asb, b, btc, c -> astc', LambdaA_new, GammaB_new, LambdaB, GammaA_new, LambdaA_new, optimize=True)
    ThetaBA_evolved = np.einsum('STst, astc -> aSTc', gate, ThetaBA, optimize=True)
    
    U2, S2, Vh2 = np.linalg.svd(ThetaBA_evolved.reshape(D * d, d * D), full_matrices=False)
    U_trunc2 = U2[:, :D].reshape(D, d, D)
    S_trunc2 = S2[:D] / np.linalg.norm(S2[:D])
    Vh_trunc2 = Vh2[:D, :].reshape(D, d, D)
    
    inv_LambdaA = np.zeros_like(LambdaA_new)
    valid = LambdaA_new > cutoff
    inv_LambdaA[valid] = 1.0 / LambdaA_new[valid]
    L_invA = np.diag(inv_LambdaA)
    
    GammaB_final = np.einsum('ab, bsc -> asc', L_invA, U_trunc2, optimize=True)
    GammaA_final = np.einsum('asb, bc -> asc', Vh_trunc2, L_invA, optimize=True)
    LambdaB_final = S_trunc2
    
    return GammaA_final, GammaB_final, LambdaA_new, LambdaB_final

def calculate_exact_energy(A, h_local) -> float:
    """Calculates exact energy using infinite transfer matrices."""
    D, d, _ = A.shape
    D_sq = D * D
    
    def matvec_R(v):
        return np.einsum('asb, bd, csd -> ac', A, v.reshape(D, D), np.conj(A), optimize=True).ravel()
    def matvec_L(v):
        return np.einsum('ac, asb, csd -> bd', v.reshape(D, D), A, np.conj(A), optimize=True).ravel()
        
    T_R = LinearOperator((D_sq, D_sq), matvec=matvec_R)
    T_L = LinearOperator((D_sq, D_sq), matvec=matvec_L)
    
    _, evecs_L = eigs(T_L, k=1, which='LR')
    _, evecs_R = eigs(T_R, k=1, which='LR')
    
    L_mat = evecs_L[:, 0].reshape(D, D)
    R_mat = evecs_R[:, 0].reshape(D, D)
    
    L_mat = (L_mat + L_mat.conj().T) / 2
    R_mat = (R_mat + R_mat.conj().T) / 2
    L_mat = L_mat / np.trace(L_mat @ R_mat)
    
    rho_left = np.einsum('lL, lsm, LSM -> mMsS', L_mat, A, np.conj(A), optimize=True)
    rho_right = np.einsum('mtr, MTR, rR -> mMtT', A, np.conj(A), R_mat, optimize=True)
    rho_2site = np.einsum('mMsS, mMtT -> sStT', rho_left, rho_right, optimize=True)
    
    E = np.einsum('STst, sStT -> ', h_local, rho_2site, optimize=True)
    return E.real

def extract_boundary_tensors(A_converged: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Extracts Left-canonical (A_L), Right-canonical (A_R), and the effective Bond matrix (C).
    Contains mathematically corrected gauge transformation order.
    """
    D, d, _ = A_converged.shape
    D_sq = D * D
    
    def matvec_R(v):
        return np.einsum('asb, bd, csd -> ac', A_converged, v.reshape(D, D), np.conj(A_converged), optimize=True).ravel()
    def matvec_L(v):
        return np.einsum('ac, asb, csd -> bd', v.reshape(D, D), A_converged, np.conj(A_converged), optimize=True).ravel()
        
    T_R = LinearOperator((D_sq, D_sq), matvec=matvec_R)
    T_L = LinearOperator((D_sq, D_sq), matvec=matvec_L)
    
    _, evecs_L = eigs(T_L, k=1, which='LR')
    _, evecs_R = eigs(T_R, k=1, which='LR')
    
    L_mat = evecs_L[:, 0].reshape(D, D)
    R_mat = evecs_R[:, 0].reshape(D, D)
    
    # Ensure Hermitian
    L_mat = (L_mat + L_mat.conj().T) / 2
    R_mat = (R_mat + R_mat.conj().T) / 2
    
    # Eigen decomposition for matrix square roots
    val_L, vec_L = np.linalg.eigh(L_mat)
    val_R, vec_R = np.linalg.eigh(R_mat)
    
    eps = 1e-15
    X = vec_L @ np.diag(np.sqrt(np.maximum(val_L, eps))) 
    Y = vec_R @ np.diag(np.sqrt(np.maximum(val_R, eps))) 
    
    X_inv = np.linalg.pinv(X)
    Y_inv = np.linalg.pinv(Y)
    
    # --- CORRECTED GAUGE TRANSFORMATIONS ---
    # Left-canonical: sum(AL^dagger AL) = I
    A_L = np.einsum('il, lpr, rj -> ipj', X, A_converged, X_inv, optimize=True)
    
    # Right-canonical: sum(AR AR^dagger) = I
    A_R = np.einsum('il, lpr, rj -> ipj', Y_inv, A_converged, Y, optimize=True)
    
    # We will also need the central bond matrix C for Phase 2 initialization
    C = X @ Y
    C /= np.linalg.norm(C)
    
    return A_L, A_R, C

if __name__ == "__main__":
    print("--- Starting Phase 1: Ground State Convergence via iTEBD ---")

    h_local = define_spin1_afh_hamiltonian()
    D_bond = 64 
    d_phys = 3
    
    print(f"Initializing 2-Site uMPS (Bond Dimension D={D_bond})...")
    GammaA, GammaB, LambdaA, LambdaB = initialize_uniform_mps(bond_dim=D_bond, phys_dim=d_phys)
    
    dt_step = 0.1
    tolerance = 1e-6 
    max_steps = 20000
    
    h_mat = h_local.reshape(d_phys * d_phys, d_phys * d_phys)
    gate = expm(-dt_step * h_mat).reshape(d_phys, d_phys, d_phys, d_phys)
    prev_energy = 0.0
    
    print("\nRunning Imaginary Time Evolution...")
    for step in range(1, max_steps + 1):
        GammaA, GammaB, LambdaA, LambdaB = run_itebd_2site(
            GammaA, GammaB, LambdaA, LambdaB, gate
        )
        
        if step % 50 == 0 or step == 1:
            energy = calculate_local_energy(GammaA, GammaB, LambdaA, LambdaB, h_local)
            eta = abs(energy - prev_energy) / dt_step
            
            print(f"   [Step {step:05d}] Energy: {energy:.8f} | eta: {eta:.2e} | dt: {dt_step}")
            
            # Dynamic time-step reduction for high precision
            if eta < 1e-4 and dt_step == 0.1:
                dt_step = 0.02
                gate = expm(-dt_step * h_mat).reshape(d_phys, d_phys, d_phys, d_phys)
            elif eta < 1e-6 and dt_step == 0.02:
                dt_step = 0.005
                gate = expm(-dt_step * h_mat).reshape(d_phys, d_phys, d_phys, d_phys)
            elif eta < 1e-9 and dt_step == 0.005:
                dt_step = 0.001
                gate = expm(-dt_step * h_mat).reshape(d_phys, d_phys, d_phys, d_phys)
                
            if eta < tolerance and dt_step == 0.001:
                print(f"   [+] Converged to target tolerance {tolerance}!")
                break
                
            prev_energy = energy

    print("\nExtracting Asymptotic Boundary Tensors (A_L, A_R)...")
    # Contract site tensors to standard form for extraction
    A_current = np.einsum('a, asb -> asb', LambdaB, GammaA)
    
    final_energy = calculate_exact_energy(A_current, h_local)
    print(f"   [+] Rigorous Final Transfer Matrix Energy: {final_energy:.8f}")
    
    A_L, A_R, C = extract_boundary_tensors(A_current)
    
    print(f"   [+] Left Canonical Tensor A_L: {A_L.shape}")
    print(f"   [+] Right Canonical Tensor A_R: {A_R.shape}")
    print(f"   [+] Center Bond Matrix C: {C.shape}")
    
    # Save these tensors for Phase 2
    np.save("A_L.npy", A_L)
    np.save("A_R.npy", A_R)
    np.save("C.npy", C)
    print("\n--- Phase 1 Complete. Tensors saved. ---")

--- Starting Phase 1: Ground State Convergence via iTEBD ---
Initializing 2-Site uMPS (Bond Dimension D=64)...

Running Imaginary Time Evolution...
   [Step 00001] Energy: -0.41550784 | eta: 4.16e+00 | dt: 0.1
   [Step 00050] Energy: -1.38724646 | eta: 9.72e+00 | dt: 0.1
   [Step 00100] Energy: -1.38850554 | eta: 1.26e-02 | dt: 0.1
   [Step 00150] Energy: -1.38852234 | eta: 1.68e-04 | dt: 0.1
   [Step 00200] Energy: -1.38852260 | eta: 2.65e-06 | dt: 0.1
   [Step 00250] Energy: -1.39892526 | eta: 5.20e-01 | dt: 0.02
   [Step 00300] Energy: -1.39894246 | eta: 8.60e-04 | dt: 0.02
   [Step 00350] Energy: -1.39894410 | eta: 8.24e-05 | dt: 0.02
   [Step 00400] Energy: -1.39894435 | eta: 1.25e-05 | dt: 0.02
   [Step 00450] Energy: -1.39894440 | eta: 2.47e-06 | dt: 0.02
   [Step 00500] Energy: -1.39894441 | eta: 6.02e-07 | dt: 0.02
   [Step 00550] Energy: -1.40083825 | eta: 3.79e-01 | dt: 0.005
   [Step 00600] Energy: -1.40084689 | eta: 1.73e-03 | dt: 0.005
   [Step 00650] Energy: -1.40084912 

In [17]:
import numpy as np

def get_spin_operators():
    """Returns the S+ and S- operators for Spin-1."""
    s2 = np.sqrt(2.0)
    Sp = np.array([[0, s2, 0], 
                   [0, 0, s2], 
                   [0, 0, 0]], dtype=complex)
    Sm = Sp.T # S- operator
    return Sp, Sm

def initialize_window(A_L: np.ndarray, A_R: np.ndarray, C: np.ndarray, N: int = 201) -> list:
    """
    Constructs the nonuniform sMPS window.
    The window spans indices 0 to 200. The orthogonality center is placed at n_c = 100.
    """
    center = N // 2
    A_window = [None] * N
    
    # Left bulk segment: completely left-canonical
    for i in range(center):
        A_window[i] = A_L.copy()
        
    # Orthogonality center: absorbs the bond matrix C
    # A_c = A_L @ C
    A_window[center] = np.einsum('isj, jk -> isk', A_L, C)
    
    # Right bulk segment: completely right-canonical
    for i in range(center + 1, N):
        A_window[i] = A_R.copy()
        
    return A_window

def apply_excitation(A_window: list, center: int):
    """
    Applies the nonunitary operator S_{m-j}^- S_{m+j}^+ at m = +/- 15, j = 5.
    Physical sites:
      m = -15 -> sites -20 and -10
      m = +15 -> sites +10 and +20
    """
    Sp, Sm = get_spin_operators()
    
    # Map physical coordinates to window array indices
    idx_m_minus_20 = center - 20
    idx_m_minus_10 = center - 10
    idx_m_plus_10  = center + 10
    idx_m_plus_20  = center + 20
    
    # Apply S- at -20 and S+ at -10
    A_window[idx_m_minus_20] = np.einsum('st, itj -> isj', Sm, A_window[idx_m_minus_20])
    A_window[idx_m_minus_10] = np.einsum('st, itj -> isj', Sp, A_window[idx_m_minus_10])
    
    # Apply S- at +10 and S+ at +20
    A_window[idx_m_plus_10]  = np.einsum('st, itj -> isj', Sm, A_window[idx_m_plus_10])
    A_window[idx_m_plus_20]  = np.einsum('st, itj -> isj', Sp, A_window[idx_m_plus_20])
    
    return A_window

def restore_mixed_canonical_gauge(A_window: list, center: int) -> list:
    """
    Sweeps through the window to restore the strict mixed-canonical gauge 
    required for Phase 3 TDVP dynamics after the nonunitary excitation.
    """
    N = len(A_window)
    
    # 1. Left-to-Right Sweep (0 to center - 1)
    for i in range(center):
        A = A_window[i]
        D_left, d, D_right = A.shape
        
        # Group left bond and physical index
        mat = A.reshape(D_left * d, D_right)
        Q, R = np.linalg.qr(mat)
        
        A_window[i] = Q.reshape(D_left, d, Q.shape[1])
        
        # Absorb R into the next tensor
        A_next = A_window[i + 1]
        A_window[i + 1] = np.einsum('ij, jsk -> isk', R, A_next)

    # 2. Right-to-Left Sweep (N-1 down to center + 1)
    for i in range(N - 1, center, -1):
        A = A_window[i]
        D_left, d, D_right = A.shape
        
        # Group physical index and right bond
        mat = A.reshape(D_left, d * D_right)
        
        # LQ decomposition via QR on transpose
        Q_T, R_T = np.linalg.qr(mat.T)
        Q = Q_T.T
        L = R_T.T
        
        A_window[i] = Q.reshape(Q.shape[0], d, D_right)
        
        # Absorb L into the previous tensor
        A_prev = A_window[i - 1]
        A_window[i - 1] = np.einsum('isj, jk -> isk', A_prev, L)

    # 3. Normalize the state at the orthogonality center
    norm_factor = np.linalg.norm(A_window[center])
    A_window[center] /= norm_factor
    print(f"   [+] Window normalized (factor: {norm_factor:.4e})")

    return A_window

if __name__ == "__main__":
    print("--- Starting Phase 2: sMPS Window Setup & Excitation ---")
    
    print("1. Loading Asymptotic Boundary Tensors...")
    try:
        A_L = np.load("A_L.npy")
        A_R = np.load("A_R.npy")
        C   = np.load("C.npy")
        print(f"   [+] A_L shape: {A_L.shape}")
    except FileNotFoundError:
        print("   [!] Error: Could not find Phase 1 tensor files. Run Phase 1 first.")
        exit()

    N_sites = 201
    center_idx = 100

    print(f"\n2. Initializing Nonuniform Window (N={N_sites})...")
    A_window = initialize_window(A_L, A_R, C, N=N_sites)

    print("\n3. Injecting Entangled Excitations (m=+/-15, j=5)...")
    A_window = apply_excitation(A_window, center_idx)

    print("\n4. Restoring Mixed-Canonical Gauge...")
    A_window = restore_mixed_canonical_gauge(A_window, center_idx)

    print("\n5. Saving Phase 2 sMPS Window...")
    A_window_array = np.array(A_window)
    np.save("sMPS_window_excited.npy", A_window_array)
    print(f"   [+] Saved 'sMPS_window_excited.npy' with shape {A_window_array.shape}")
    print("--- Phase 2 Complete ---")

--- Starting Phase 2: sMPS Window Setup & Excitation ---
1. Loading Asymptotic Boundary Tensors...
   [+] A_L shape: (64, 3, 64)

2. Initializing Nonuniform Window (N=201)...

3. Injecting Entangled Excitations (m=+/-15, j=5)...

4. Restoring Mixed-Canonical Gauge...
   [+] Window normalized (factor: 1.7778e+00)

5. Saving Phase 2 sMPS Window...
   [+] Saved 'sMPS_window_excited.npy' with shape (201, 64, 3, 64)
--- Phase 2 Complete ---


In [18]:
import numpy as np
from tqdm import tqdm

def get_sz():
    """Returns the S^z operator for Spin-1."""
    return np.diag([1.0, 0.0, -1.0]).astype(complex)

def check_isometry_constraints(A_window: np.ndarray, center: int = 100, tol: float = 1e-11):
    """
    Scans the window to verify strict Left and Right canonical gauges.
    """
    N, D, d, _ = A_window.shape
    I_exact = np.eye(D, dtype=complex)
    
    max_left_error = 0.0
    max_right_error = 0.0
    
    # 1. Check Left-Canonical Gauge (n < center)
    for n in range(center):
        A = A_window[n]
        # sum_s A_s^H A_s = I  =>  np.einsum('isj, isk -> jk', A*, A)
        L_metric = np.einsum('isj, isk -> jk', A.conj(), A)
        err = np.linalg.norm(L_metric - I_exact)
        if err > max_left_error: max_left_error = err
        
        if err > tol:
            print(f"   [!] GAUGE FAILURE at Left Site {n}. Error: {err:.2e}")
            return False

    # 2. Check Right-Canonical Gauge (n > center)
    for n in range(center + 1, N):
        A = A_window[n]
        # sum_s A_s A_s^H = I  =>  np.einsum('isj, ksj -> ik', A, A*)
        R_metric = np.einsum('isj, ksj -> ik', A, A.conj())
        err = np.linalg.norm(R_metric - I_exact)
        if err > max_right_error: max_right_error = err
        
        if err > tol:
            print(f"   [!] GAUGE FAILURE at Right Site {n}. Error: {err:.2e}")
            return False
            
    # 3. Check Center Normalization
    norm_c = np.linalg.norm(A_window[center])
    
    print(f"   [+] Left-Canonical Max Error:  {max_left_error:.4e}")
    print(f"   [+] Right-Canonical Max Error: {max_right_error:.4e}")
    print(f"   [+] State Normalization (n_c): {norm_c:.6f} (Target: 1.000000)")
    
    return True

def measure_rigorous_sz_profile(A_window: np.ndarray):
    """
    Calculates exact <S^z_n> by performing a full sweep of the 
    density environments. Progress tracked via tqdm, and tensor 
    contractions explicitly optimized to prevent O(D^4) memory bottlenecks.
    """
    N, D, d, _ = A_window.shape
    Sz = get_sz()
    profile = np.zeros(N)
    
    L = [None] * N
    R = [None] * N
    
    # Initialize extreme boundaries with Identity
    L[0] = np.eye(D, dtype=complex)
    R[N-1] = np.eye(D, dtype=complex)
    
    print("   [*] Calculating Left Environments...")
    # Forward Sweep (Calculate L_n)
    for n in tqdm(range(0, N - 1), desc="Left Sweep", leave=False):
        A = A_window[n]
        # L_{n+1}^{j,l} = sum_{i,k,s} A*_{i,s,j} A_{k,s,l} L_n^{i,k}
        # optimize=True is CRITICAL here for D=64
        L[n+1] = np.einsum('isj, ksl, ik -> jl', A.conj(), A, L[n], optimize=True)
        
    print("   [*] Calculating Right Environments...")
    # Backward Sweep (Calculate R_n)
    for n in tqdm(range(N - 1, 0, -1), desc="Right Sweep", leave=False):
        A = A_window[n]
        # R_{n-1}^{i,k} = sum_{j,l,s} A*_{i,s,j} A_{k,s,l} R_n^{j,l}
        R[n-1] = np.einsum('isj, ksl, jl -> ik', A.conj(), A, R[n], optimize=True)

    print("   [*] Contracting Local Observables...")
    # Measure Local Observables
    for n in tqdm(range(N), desc="Measurements", leave=False):
        A = A_window[n]
        # <psi | S^z_n | psi> = sum L_n^{i,k} A*_{i,s,j} S^z_{s,t} A_{k,t,l} R_n^{j,l}
        val = np.einsum('ik, isj, st, ktl, jl -> ', L[n], A.conj(), Sz, A, R[n], optimize=True)
        profile[n] = val.real
        
    return profile

if __name__ == "__main__":
    print("--- Phase 2.5: sMPS Manifold Sanity Check ---")
    
    try:
        A_window = np.load("sMPS_window_excited.npy")
        print(f"1. Loaded sMPS array of shape: {A_window.shape}")
    except FileNotFoundError:
        print("[!] Error: sMPS_window_excited.npy not found.")
        exit()
        
    center_idx = 100
    
    print("\n2. Verifying Mixed-Canonical Gauge Constraints...")
    passed = check_isometry_constraints(A_window, center=center_idx)
    
    if passed:
        print("\n3. Measuring Exact <S^z> Profile...")
        sz_profile = measure_rigorous_sz_profile(A_window)
        
        print("   [+] Top 5 Excitations by magnitude:")
        # Sort by absolute magnitude and get top 5
        top_indices = np.argsort(np.abs(sz_profile))[-5:][::-1]
        
        for idx in top_indices:
            site = idx - center_idx # Map 0..200 to -100..100
            print(f"       Site m={site:4d} | <S^z> = {sz_profile[idx]:.6f}")
            
        print("\n   [Diagnosis] If the top sites are exactly m = -20, -10, +10, +20")
        print("   [Diagnosis] and all gauge errors are < 1e-11, the state is flawless.")
        print("--- All Checks Passed. Ready for Phase 3. ---")

--- Phase 2.5: sMPS Manifold Sanity Check ---
1. Loaded sMPS array of shape: (201, 64, 3, 64)

2. Verifying Mixed-Canonical Gauge Constraints...
   [+] Left-Canonical Max Error:  5.4401e-15
   [+] Right-Canonical Max Error: 6.4705e-15
   [+] State Normalization (n_c): 1.000000 (Target: 1.000000)

3. Measuring Exact <S^z> Profile...
   [*] Calculating Left Environments...


   [*] Calculating Right Environments...


   [*] Contracting Local Observables...


   [+] Top 5 Excitations by magnitude:
       Site m=  20 | <S^z> = 0.500002
       Site m= -10 | <S^z> = 0.500002
       Site m= -20 | <S^z> = -0.500002
       Site m=  10 | <S^z> = -0.500002
       Site m= -19 | <S^z> = 0.268622

   [Diagnosis] If the top sites are exactly m = -20, -10, +10, +20
   [Diagnosis] and all gauge errors are < 1e-11, the state is flawless.
--- All Checks Passed. Ready for Phase 3. ---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# MPO & Observables Setup
# ─────────────────────────────────────────────────────────────────────────────

def get_spin_operators():
    s2 = np.sqrt(2.0)
    Sp = np.array([[0, s2, 0], [0, 0, s2], [0, 0, 0]], dtype=complex)
    Sm = Sp.T
    Sx = (Sp + Sm) / 2.0
    Sy = (Sp - Sm) / 2.0j
    Sz = np.diag([1., 0., -1.]).astype(complex)
    return Sx, Sy, Sz

def build_mpo_array(N: int, e_bulk: float):
    Sx, Sy, Sz = get_spin_operators()
    I = np.eye(3, dtype=complex)

    W_base = np.zeros((5, 5, 3, 3), dtype=complex)
    W_base[0, 0] = I
    W_base[4, 4] = I
    W_base[1, 0] = Sx; W_base[2, 0] = Sy; W_base[3, 0] = Sz
    W_base[4, 1] = Sx; W_base[4, 2] = Sy; W_base[4, 3] = Sz
    W_base[4, 0] = -e_bulk * I

    W_array = []
    for n in range(N):
        W_n = W_base.copy()
        eps_n = np.exp(-((n - 10)**2) / 18.0) + np.exp(-((n - 190)**2) / 18.0)
        damping = 1.0 - 1.0j * eps_n

        W_n[4, 1] *= damping
        W_n[4, 2] *= damping
        W_n[4, 3] *= damping
        W_array.append(W_n)
    return W_array

# ─────────────────────────────────────────────────────────────────────────────
# BLAS-Optimized Environment Contractions
# ─────────────────────────────────────────────────────────────────────────────

def get_environments(A_win: list, W_arr: list, D: int):
    N = len(A_win)
    L = [None] * (N + 1)
    R = [None] * (N + 1)

    L[0] = np.zeros((5, D, D), dtype=complex)
    L[0][4] = np.eye(D, dtype=complex)
    R[N] = np.zeros((5, D, D), dtype=complex)
    R[N][0] = np.eye(D, dtype=complex)

    # Left Sweep
    for n in range(N):
        A = A_win[n]
        L_A = np.tensordot(L[n], A, axes=([2], [0]))                   # (a, i, t, l)
        L_AW = np.tensordot(L_A, W_arr[n], axes=([0, 2], [0, 3]))       # (i, l, b, s)
        L_next = np.tensordot(L_AW, A.conj(), axes=([0, 3], [0, 1]))     # (l, b, j)
        L[n+1] = L_next.transpose(1, 2, 0)                             # (b, j, l)

    # Right Sweep
    for n in range(N - 1, -1, -1):
        A = A_win[n]
        R_A = np.tensordot(R[n+1], A, axes=([2], [2]))                   # (b, j, k, t)
        R_AW = np.tensordot(R_A, W_arr[n], axes=([0, 3], [1, 3]))         # (j, k, a, s)
        R_next = np.tensordot(R_AW, A.conj(), axes=([0, 3], [2, 1]))       # (k, a, i)
        R[n] = R_next.transpose(1, 2, 0)                                 # (a, i, k)

    return L, R

def get_tangent_vector(A_win: list, W_arr: list, center: int):
    N = len(A_win)
    D = A_win[0].shape[0]
    L, R = get_environments(A_win, W_arr, D)
    B_win = [None] * N

    for n in range(N):
        A = A_win[n]
        # Force: F_n = L * W * A * R
        L_A = np.tensordot(L[n], A, axes=([2], [0]))                     # (a, i, t, l)
        L_AW = np.tensordot(L_A, W_arr[n], axes=([0, 2], [0, 3]))         # (i, l, b, s)
        F_n = np.tensordot(L_AW, R[n+1], axes=([1, 2], [2, 0]))           # (i, s, j)

        B = -1.0j * F_n

        # Tangent Space Gauge Projection
        if n < center:
            X = np.tensordot(A.conj(), B, axes=([0, 1], [0, 1]))          # (j, j)
            B -= np.tensordot(A, X, axes=([2], [0]))                      # (i, s, j)
        elif n > center:
            X = np.tensordot(B, A.conj(), axes=([1, 2], [1, 2]))          # (i, i)
            B -= np.tensordot(X, A, axes=([1], [0]))                      # (i, s, j)

        B_win[n] = B
    return B_win

# ─────────────────────────────────────────────────────────────────────────────
# Gauge Restoration & RK4 Evolution
# ─────────────────────────────────────────────────────────────────────────────

def restore_mixed_canonical_gauge(A_window: list, center: int):
    N = len(A_window)
    A_win = [A.copy() for A in A_window]

    for i in range(center):
        A = A_win[i]
        D_l, d, D_r = A.shape
        Q, R_mat = np.linalg.qr(A.reshape(D_l * d, D_r))
        A_win[i] = Q.reshape(D_l, d, Q.shape[1])
        A_win[i+1] = np.tensordot(R_mat, A_win[i+1], axes=([1], [0]))

    for i in range(N - 1, center, -1):
        A = A_win[i]
        D_l, d, D_r = A.shape
        Q_T, L_mat_T = np.linalg.qr(A.reshape(D_l, d * D_r).T)
        A_win[i] = Q_T.T.reshape(Q_T.shape[1], d, D_r)
        A_win[i-1] = np.tensordot(A_win[i-1], L_mat_T.T, axes=([2], [0]))

    A_win[center] /= np.linalg.norm(A_win[center])
    return A_win

def step_rk4(A_win: list, W_arr: list, center: int, dt: float):
    def add_states(A1, A2, factor):
        return [a1 + factor * a2 for a1, a2 in zip(A1, A2)]

    K1 = get_tangent_vector(A_win, W_arr, center)
    A_k2 = restore_mixed_canonical_gauge(add_states(A_win, K1, 0.5 * dt), center)

    K2 = get_tangent_vector(A_k2, W_arr, center)
    A_k3 = restore_mixed_canonical_gauge(add_states(A_win, K2, 0.5 * dt), center)

    K3 = get_tangent_vector(A_k3, W_arr, center)
    A_k4 = restore_mixed_canonical_gauge(add_states(A_win, K3, dt), center)

    K4 = get_tangent_vector(A_k4, W_arr, center)

    A_next = A_win.copy()
    for n in range(len(A_win)):
        A_next[n] += (dt / 6.0) * (K1[n] + 2*K2[n] + 2*K3[n] + K4[n])

    return restore_mixed_canonical_gauge(A_next, center)

def calculate_Sz_profile(A_window: list):
    """
    Robust Observable Measurement. Uses einsum with optimize=True
    to guarantee flawless index routing and prevent tuple errors.
    """
    N = len(A_window)
    D = A_window[0].shape[0]
    _, _, Sz = get_spin_operators()
    profile = np.zeros(N)

    L = [None] * N
    R = [None] * N
    L[0] = np.eye(D, dtype=complex)
    R[N-1] = np.eye(D, dtype=complex)

    for n in range(0, N - 1):
        A = A_window[n]
        L[n+1] = np.einsum('isj, ksl, ik -> jl', A.conj(), A, L[n], optimize=True)

    for n in range(N - 1, 0, -1):
        A = A_window[n]
        R[n-1] = np.einsum('isj, ksl, jl -> ik', A.conj(), A, R[n], optimize=True)

    for n in range(N):
        A = A_window[n]
        val = np.einsum('ik, isj, st, ktl, jl -> ', L[n], A.conj(), Sz, A, R[n], optimize=True)
        profile[n] = val.real

    return profile

# ─────────────────────────────────────────────────────────────────────────────
# Main Execution Loop
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("--- Phase 3: True TDVP Real-Time Scattering (RK4) ---")
    
    A_L = np.load("A_L.npy")
    C = np.load("C.npy")
    A_window = list(np.load("sMPS_window_excited.npy"))
    N_sites = 201
    center_idx = 100
    
    _, _, Sz = get_spin_operators()
    s2 = np.sqrt(2.0); Sp = np.array([[0, s2, 0], [0, 0, s2], [0, 0, 0]], dtype=complex)
    Sx, Sy = (Sp + Sp.T)/2.0, (Sp - Sp.T)/2.0j
    h_AFH = (np.kron(Sx, Sx) + np.kron(Sy, Sy) + np.kron(Sz, Sz)).real.reshape(3,3,3,3)
    
    # Exact Bulk Energy 
    R_mat = C @ C.conj().T
    Theta_L = np.tensordot(A_L, A_L, axes=([2], [0]))                     
    E_term1 = np.tensordot(Theta_L.conj(), h_AFH, axes=([1, 2], [0, 1]))  
    E_term2 = np.tensordot(E_term1, Theta_L, axes=([0, 2, 3], [0, 1, 2])) 
    e_bulk = np.sum(E_term2 * R_mat.T).real
    
    print(f"   [+] Validated Bulk Energy Shift: {e_bulk:.6f}")
    
    W_array = build_mpo_array(N_sites, e_bulk)
    
    dt = 0.025 # Reduced dt for tangent-space stability
    t_max = 15.0
    steps = int(t_max / dt)
    history_Sz = []
    
    out_dir = "phase3_output"
    os.makedirs(out_dir, exist_ok=True)
    
    print("\n   [!] Initiating True TDVP RK4 Integrator.")
    
    for step in tqdm(range(steps + 1), desc="Evolving Manifold", unit="step"):
        t = step * dt
        Sz_profile = calculate_Sz_profile(A_window)
        history_Sz.append(Sz_profile)
        
        # Log the top 4 peaks every 10 steps to track the wavepacket
        if step % 10 == 0:
            top_indices = np.argsort(np.abs(Sz_profile))[-4:][::-1]
            peaks_info = " | ".join([f"Site {idx - center_idx}: {Sz_profile[idx]:.4f}" for idx in top_indices])
            tqdm.write(f"   --> t = {t:5.2f} | Peaks: {peaks_info}")
            
        if step < steps:
            A_window = step_rk4(A_window, W_array, center_idx, dt)
            
    print("\n--- Evolution Complete. Generating Final Plot ---")
    
    history_Sz = np.array(history_Sz)
    plt.figure(figsize=(10, 6))
    extent = [-100, 100, t_max, 0] 
    plt.imshow(history_Sz, aspect='auto', cmap='jet', extent=extent, vmin=-0.5, vmax=0.5)
    plt.colorbar(label=r'$\langle S_n^z \rangle$')
    plt.xlabel('Site $n$')
    plt.ylabel('Time $t (\hbar/J)$')
    plt.title('True TDVP Scattering of Entangled Excitations (Spin-1 AFH)')
    
    plot_path = os.path.join(out_dir, "Figure2_Scattering_TDVP.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    np.save(os.path.join(out_dir, "Sz_history_tdvp.npy"), history_Sz)
    
    print(f"   [+] Saved accurate scattering heatmap to {plot_path}")